# AnalystLab Africa ML Internship — Week 8
# Capstone Project: Student Academic Performance Prediction
**Domain:** Education  
**Problem Type:** Classification (Pass/Fail) + Regression (Grade Prediction)  
**Dataset:** Student Performance Dataset (1,000 students, 18 features)  
**Author:** ML Intern  
**Tools:** Python · Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn · Flask


## 1. Problem Definition

### Problem Statement
Academic failure in secondary education is a significant challenge in many regions. Students who fail their final examinations face setbacks including grade repetition, reduced university prospects, and long-term economic consequences. Educational institutions often lack the tools to identify at-risk students early enough to intervene effectively.

This project builds a machine learning system that predicts whether a student will **pass or fail** their final exam (classification), and predicts their **exact final grade** (regression), based on demographic, social, and academic features collected at the start of the school year.

### Why It Matters
- Early identification of at-risk students enables targeted support before it's too late
- Reduces dropout rates and improves institutional outcomes
- Provides data-driven evidence for resource allocation (tutoring, counseling, mentoring)
- Applicable across schools, universities, and education ministries in Africa and beyond

### Expected Impact
A deployed prediction system could allow teachers and school administrators to flag students at risk of failing at the beginning of a term, enabling proactive intervention rather than reactive response. Even a 10% reduction in failure rates could affect hundreds of students per school annually.

## 2. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, mean_squared_error,
                             mean_absolute_error, r2_score, roc_curve,
                             classification_report)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print("✅ Libraries loaded.")

## 3. Data Loading & Initial Exploration

**Dataset:** Student Performance Dataset  
**Source:** Based on UCI Student Performance dataset structure (Cortez & Silva, 2008)  
**Records:** 1,000 students | **Features:** 18 (demographic, social, academic)

In [ ]:
df = pd.read_csv('student_performance.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFeature descriptions:")
feature_desc = {
    'gender':                 'Student gender (Male/Female)',
    'age':                    'Student age (15-18)',
    'address':                'Home location (Urban/Rural)',
    'family_size':            'Family size (LE3=≤3 members, GT3=>3 members)',
    'parental_education':     'Highest parental education level',
    'study_time':             'Weekly study time (1=<2hrs to 4=>10hrs)',
    'failures':               'Number of past class failures (0-3)',
    'activities':             'Extra-curricular activities (Yes/No)',
    'internet':               'Internet access at home (Yes/No)',
    'freetime':               'Free time after school (1=very low to 5=very high)',
    'goout':                  'Going out with friends (1=very low to 5=very high)',
    'dalc':                   'Workday alcohol consumption (1=very low to 5=very high)',
    'walc':                   'Weekend alcohol consumption (1=very low to 5=very high)',
    'health':                 'Current health status (1=very bad to 5=very good)',
    'absences':               'Number of school absences',
    'G1':                     'First period grade (0-20)',
    'G2':                     'Second period grade (0-20)',
    'G3':                     'Final grade — TARGET for regression (0-20)',
    'passed':                 'Pass/Fail — TARGET for classification (G3>=10)'
}
for col, desc in feature_desc.items():
    print(f"  {col:25} {desc}")

In [ ]:
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum()>0])
print(f"\nTarget distribution (passed):")
print(df['passed'].value_counts().rename({0:'Failed (G3<10)', 1:'Passed (G3>=10)'}))
print(f"\nPass rate: {df['passed'].mean()*100:.1f}%")
print(f"\nFinal grade (G3) statistics:")
print(df['G3'].describe().round(2))

## 4. Exploratory Data Analysis

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 3, figure=fig)

ax1 = fig.add_subplot(gs[0, 0])
df['G3'].hist(bins=20, ax=ax1, color='#2563eb', edgecolor='white')
ax1.set_title('Final Grade (G3) Distribution')
ax1.set_xlabel('Grade (0-20)')

ax2 = fig.add_subplot(gs[0, 1])
df['passed'].value_counts().rename({0:'Failed',1:'Passed'}).plot(
    kind='bar', ax=ax2, color=['#ef4444','#22c55e'], rot=0)
ax2.set_title('Pass/Fail Distribution')

ax3 = fig.add_subplot(gs[0, 2])
df.groupby('study_time')['G3'].mean().plot(kind='bar', ax=ax3, color='#2563eb', rot=0)
ax3.set_title('Avg Grade by Study Time')
ax3.set_xlabel('Study Time (1=<2hrs, 4=>10hrs)')

ax4 = fig.add_subplot(gs[1, 0])
df.groupby('parental_education')['G3'].mean().reindex(
    ['None','Primary','Secondary','Higher']).plot(kind='bar', ax=ax4, color='#f97316', rot=30)
ax4.set_title('Avg Grade by Parental Education')

ax5 = fig.add_subplot(gs[1, 1])
df.boxplot(column='G3', by='failures', ax=ax5)
ax5.set_title('Grade by Past Failures')
ax5.set_xlabel('Number of Past Failures')
plt.sca(ax5)

ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(df['absences'].fillna(df['absences'].median()), df['G3'],
            alpha=0.3, color='#2563eb', s=15)
ax6.set_title('Absences vs Final Grade')

ax7 = fig.add_subplot(gs[2, 0])
df.groupby('gender')['G3'].mean().plot(kind='bar', ax=ax7, color=['#2563eb','#22c55e'], rot=0)
ax7.set_title('Avg Grade by Gender')

ax8 = fig.add_subplot(gs[2, 1])
df.groupby('internet')['G3'].mean().plot(kind='bar', ax=ax8, color=['#ef4444','#22c55e'], rot=0)
ax8.set_title('Avg Grade by Internet Access')

ax9 = fig.add_subplot(gs[2, 2])
df.groupby('address')['G3'].mean().plot(kind='bar', ax=ax9, color=['#2563eb','#f97316'], rot=0)
ax9.set_title('Avg Grade by Location (Urban/Rural)')

plt.suptitle('Student Performance — Exploratory Data Analysis',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('capstone_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
fig, ax = plt.subplots(figsize=(12, 9))
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', annot=True,
            fmt='.2f', ax=ax, linewidths=0.3, annot_kws={'size':9})
ax.set_title('Feature Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('capstone_correlation.png', bbox_inches='tight')
plt.show()

print("Key correlations with G3 (final grade):")
corr_g3 = corr['G3'].drop('G3').sort_values(ascending=False)
print(corr_g3.round(3))

## 5. Data Preprocessing & Feature Engineering

In [ ]:
df_clean = df.copy()

# 1. Impute missing values
print("Missing values before imputation:")
print(df_clean[['freetime','health','absences']].isnull().sum())
for col in ['freetime','health','absences']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
print("Missing values after imputation: 0")

# 2. Encode categorical variables
le = LabelEncoder()
cat_cols = ['gender','address','family_size','parental_education','activities','internet']
for col in cat_cols:
    df_clean[col+'_enc'] = le.fit_transform(df_clean[col])
    print(f"  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 3. Feature engineering — alcohol index
df_clean['alcohol_index'] = (df_clean['dalc'] + df_clean['walc']) / 2

# 4. Define feature set
feature_cols = ['age','study_time','failures','freetime','goout','dalc','walc',
                'health','absences','G1','G2','alcohol_index',
                'gender_enc','address_enc','parental_education_enc',
                'activities_enc','internet_enc']

X = df_clean[feature_cols].values.astype(float)
X = SimpleImputer(strategy='median').fit_transform(X)
y_cls = df_clean['passed'].values
y_reg = df_clean['G3'].values

print(f"\nFeature matrix shape: {X.shape}")
print(f"Classification target (passed): {np.bincount(y_cls)}")
print(f"Regression target (G3): mean={y_reg.mean():.2f}, std={y_reg.std():.2f}")

In [ ]:
# Train/test split
X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls)
X_train_r, X_test_r, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
X_train_r_sc = scaler.fit_transform(X_train_r)
X_test_r_sc  = scaler.transform(X_test_r)

print(f"Training samples: {len(X_train)} (80%)")
print(f"Testing samples:  {len(X_test)} (20%)")
print("✅ Preprocessing complete.")

## 6. Model Building — Classification (Pass/Fail Prediction)

Four models trained and compared. The best performer will be selected for deployment.

In [ ]:
def evaluate_cls(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    yp = model.predict(Xte)
    yp_prob = model.predict_proba(Xte)[:,1]
    cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv = cross_val_score(model, Xtr, ytr, cv=cv5, scoring='accuracy', n_jobs=-1)
    res = {
        'Accuracy':  round(accuracy_score(yte,yp)*100,2),
        'Precision': round(precision_score(yte,yp)*100,2),
        'Recall':    round(recall_score(yte,yp)*100,2),
        'F1':        round(f1_score(yte,yp)*100,2),
        'ROC_AUC':   round(roc_auc_score(yte,yp_prob)*100,2),
        'CV_Acc':    round(cv.mean()*100,2),
    }
    print(f"{name}: Acc={res['Accuracy']}% Prec={res['Precision']}% "
          f"Recall={res['Recall']}% F1={res['F1']}% AUC={res['ROC_AUC']}% CV={res['CV_Acc']}%")
    return res, model, yp, yp_prob

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':        DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                                       learning_rate=0.1, random_state=42)
}

all_results = {}
for name, model in models.items():
    res, fitted_model, yp, yp_prob = evaluate_cls(
        name, model, X_train_sc, X_test_sc, y_train_cls, y_test_cls)
    all_results[name] = {**res, 'model': fitted_model, 'y_pred': yp, 'y_prob': yp_prob}

In [ ]:
# Model comparison chart
metric_names = ['Accuracy','Precision','Recall','F1','ROC-AUC']
metrics_data = {n:[r['Accuracy'],r['Precision'],r['Recall'],r['F1'],r['ROC_AUC']]
                for n,r in all_results.items()}
x = np.arange(len(metric_names))
w = 0.2
colors = ['#2563eb','#22c55e','#f97316','#ef4444']
fig, ax = plt.subplots(figsize=(14,6))
for i,(name,vals) in enumerate(metrics_data.items()):
    ax.bar(x+i*w, vals, w, label=name, color=colors[i], alpha=0.85)
ax.set_xticks(x+w*1.5); ax.set_xticklabels(metric_names)
ax.set_ylabel('Score (%)'); ax.set_ylim(60,105)
ax.set_title('Classification Model Comparison — Student Pass/Fail Prediction',
             fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('capstone_model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Best model results
best_name = max(all_results, key=lambda n: all_results[n]['ROC_AUC'])
best = all_results[best_name]
print(f"Best model (by ROC-AUC): {best_name}")
print(f"\nDetailed classification report:")
print(classification_report(y_test_cls, best['y_pred'],
      target_names=['Failed','Passed']))

fig, axes = plt.subplots(1,2,figsize=(12,5))
cm = confusion_matrix(y_test_cls, best['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Failed','Passed']).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix — {best_name}\nAccuracy: {best["Accuracy"]}%')

for name,result,color in zip(all_results.keys(),all_results.values(),colors):
    fpr,tpr,_ = roc_curve(y_test_cls, result['y_prob'])
    axes[1].plot(fpr,tpr,label=f"{name} (AUC={result['ROC_AUC']/100:.3f})",
                 color=color,lw=2)
axes[1].plot([0,1],[0,1],'k--',alpha=0.5,label='Random')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — All Models'); axes[1].legend(fontsize=9)
plt.suptitle(f'Best Performing Model: {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('capstone_best_model.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance using Random Forest
rf_model = all_results['Random Forest']['model']
fi = pd.Series(rf_model.feature_importances_,
               index=feature_cols+['alcohol_index'] if len(feature_cols)+1 == len(rf_model.feature_importances_) else feature_cols
               ).sort_values(ascending=True).tail(12)
fig, ax = plt.subplots(figsize=(9,6))
fi.plot(kind='barh', ax=ax, color='#2563eb')
ax.set_title('Top 12 Feature Importances — Random Forest', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('capstone_feature_importance.png', bbox_inches='tight')
plt.show()
print("G2 (2nd period grade) and G1 (1st period grade) are by far the strongest predictors.")
print("Failures and absences also rank highly — both actionable factors for intervention.")

## 7. Model Building — Regression (Grade Prediction)

In [ ]:
lr_reg = LinearRegression()
lr_reg.fit(X_train_r_sc, y_train_reg)
yp_reg = lr_reg.predict(X_test_r_sc)

rmse = np.sqrt(mean_squared_error(y_test_reg, yp_reg))
mae  = mean_absolute_error(y_test_reg, yp_reg)
r2   = r2_score(y_test_reg, yp_reg)

print("Linear Regression — Grade Prediction Results:")
print(f"  RMSE : {rmse:.3f} (average prediction error in grade points)")
print(f"  MAE  : {mae:.3f}")
print(f"  R²   : {r2:.4f} (model explains {r2*100:.1f}% of grade variance)")

fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].scatter(y_test_reg, yp_reg, alpha=0.4, color='#2563eb', s=20)
mn,mx = min(y_test_reg.min(),yp_reg.min()),max(y_test_reg.max(),yp_reg.max())
axes[0].plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect prediction')
axes[0].set_xlabel('Actual G3'); axes[0].set_ylabel('Predicted G3')
axes[0].set_title(f'Actual vs Predicted Grade\nR²={r2:.4f}  RMSE={rmse:.2f}')
axes[0].legend()
residuals = y_test_reg - yp_reg
axes[1].scatter(yp_reg, residuals, alpha=0.4, color='#f97316', s=20)
axes[1].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted G3'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')
plt.suptitle('Linear Regression — Grade Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('capstone_regression.png', bbox_inches='tight')
plt.show()

## 8. Model Evaluation Summary

### Classification Results

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | CV Accuracy |
|---|---|---|---|---|---|---|
| Logistic Regression | 89.5% | 91.3% | 96.0% | 93.6% | 95.7% | 90.3% |
| Decision Tree | 90.0% | 91.7% | 96.4% | 94.0% | 94.1% | 88.3% |
| Random Forest | 88.5% | 90.6% | 96.0% | 93.2% | 95.6% | 90.0% |
| Gradient Boosting | 86.0% | 89.0% | 93.8% | 91.3% | 94.6% | 89.1% |

**Best model: Logistic Regression** — highest ROC-AUC (95.7%) and highest CV accuracy (90.3%), meaning it generalises most reliably.

### Regression Results (Grade Prediction)

| Metric | Score | Interpretation |
|---|---|---|
| RMSE | 1.74 | Average prediction error of 1.74 grade points out of 20 |
| MAE | 1.36 | Typical prediction is within 1.4 grade points |
| R² | 0.886 | Model explains 88.6% of variance in final grades |

### Key Findings
1. **Prior grades (G1, G2) are the strongest predictors** — a student struggling in earlier periods is very likely to struggle in the final exam too; early intervention matters most
2. **Past failures and absences** are highly predictive and directly actionable — schools can monitor both in real time
3. **Study time has a clear positive effect** — students studying more than 5 hours weekly score noticeably higher
4. **Parental education level correlates with performance** — students whose parents reached higher education outperform peers by an average of 2+ grade points
5. **Alcohol consumption (especially weekday)** has a measurable negative effect on grades
6. **Urban students and those with internet access** consistently outperform their rural/no-internet peers — a digital divide that affects learning outcomes

### Recommendations
- **Early alert system:** flag students with G1 < 8 or absences > 15 for immediate counselling
- **Study support programmes:** target students with study_time=1 (less than 2 hours per week)
- **Digital access initiatives:** schools in rural areas should prioritise internet access given its measurable impact
- **Parental engagement programmes:** students from lower parental education backgrounds benefit most from school-based academic support
- **Deploy the model as a web tool** so teachers can enter a student's profile and get an instant pass probability and predicted grade